# Voice Processing — ECAPA-TDNN embeddings


# PROCESAMIENTO DE LA VOZ

In [20]:
! pip install soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------  1.0/1.0 MB 66.9 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 32.6 MB/s eta 0:00:00


# Voice Processing

## Objective

Phase 1 produced a **32-dimensional EEG fingerprint** per subject (via EEGNet). Phase 3 will map this EEG fingerprint to a **voice identity**, so that a person who cannot speak can be given a voice that matches their neural signature.

To do this, we need a parallel **voice fingerprint** — a compact vector that captures *who* a person sounds like, independent of *what* they say. This section processes the BIOMEX voice recordings to extract such a representation.

## VOICE-0: Data inventory

Before processing anything, we inspect the voice data to understand its format, structure, and completeness — the same approach used for EEG exploration in Part 1.

In [21]:
# ============================================================
# CELL VOICE-0: Inventory of the Voice data in BIOMEX
# Read-only — just inspects what audio files exist and their
# format, before deciding on a processing pipeline.
# ============================================================

import soundfile as sf   # pip install soundfile if missing
from pathlib import Path

print("=== Voice folder structure for F001 ===")
voice_path = BIOMEX_ROOT / "F001" / "F001" / "Voice"

if not voice_path.exists():
    # Try to locate the Voice folder wherever it is
    print(f"  Not at expected path. Searching under F001...")
    for p in (BIOMEX_ROOT / "F001").rglob("*"):
        if p.is_dir() and 'voice' in p.name.lower():
            voice_path = p
            print(f"  Found: {p}")
            break

# List all files in F001's Voice folder
print(f"\n  Contents of {voice_path}:")
files = sorted(voice_path.iterdir()) if voice_path.exists() else []
ext_count = {}
for f in files:
    ext = f.suffix.lower()
    ext_count[ext] = ext_count.get(ext, 0) + 1
for ext, n in sorted(ext_count.items()):
    print(f"    {ext}: {n} files")

print(f"\n  First 10 files:")
for f in files[:10]:
    size_kb = f.stat().st_size / 1024
    print(f"    {f.name:45s}  {size_kb:8.1f} KB")

# ── Inspect one WAV file in detail ────────────────────────────────
wav_files = [f for f in files if f.suffix.lower() == '.wav']
if wav_files:
    print(f"\n=== Audio properties of {wav_files[0].name} ===")
    data, samplerate = sf.read(wav_files[0])
    duration = len(data) / samplerate
    print(f"    Sample rate : {samplerate} Hz")
    print(f"    Duration    : {duration:.2f} s")
    print(f"    Samples     : {len(data)}")
    print(f"    Channels    : {1 if data.ndim == 1 else data.shape[1]}")
    print(f"    Amplitude   : min={data.min():.3f}  max={data.max():.3f}")

# ── Inspect one .lab file (likely transcription or timing) ───────
lab_files = [f for f in files if f.suffix.lower() == '.lab']
if lab_files:
    print(f"\n=== Contents of {lab_files[0].name} ===")
    with open(lab_files[0], encoding='utf-8', errors='ignore') as fh:
        content = fh.read()
    print(f"    (first 500 chars)")
    print("    " + content[:500].replace('\n', '\n    '))

# ── Count WAVs per subject across the whole dataset ──────────────
print(f"\n=== WAV count per subject (first 10 subjects) ===")
for sid in metadata['subject_id'].values[:10]:
    vp = BIOMEX_ROOT / sid / sid / "Voice"
    n_wav = len(list(vp.glob("*.wav"))) if vp.exists() else 0
    print(f"    {sid}: {n_wav} wav files")

=== Voice folder structure for F001 ===

  Contents of C:\Users\Laura\OneDrive\Escritorio\TFM_part1\BIOMEX\BIOMEX\F001\F001\Voice:
    .lab: 20 files
    .wav: 20 files

  First 10 files:
    F001_01G04_1.lab                                    0.2 KB
    F001_01G04_1.wav                                  250.0 KB
    F001_01G04_10.lab                                   0.2 KB
    F001_01G04_10.wav                                 250.0 KB
    F001_01G04_2.lab                                    0.2 KB
    F001_01G04_2.wav                                  250.0 KB
    F001_01G04_3.lab                                    0.2 KB
    F001_01G04_3.wav                                  250.0 KB
    F001_01G04_4.lab                                    0.2 KB
    F001_01G04_4.wav                                  250.0 KB

=== Audio properties of F001_01G04_1.wav ===
    Sample rate : 16000 Hz
    Duration    : 8.00 s
    Samples     : 128001
    Channels    : 1
    Amplitude   : min=-0.161  max=0.206

### VOICE-0 Results: Data inventory

| Property | Value |
|----------|-------|
| Files per subject | 20 (10 .wav + 10 .lab) |
| Sessions | G04 (4-digit sequences) × 10 clips + G10 (10-digit sequences) × 10 clips |
| Sample rate | 16,000 Hz (standard for speech) |
| Duration per clip | 8 s (G04) to 20 s (G10) |
| Channels | Mono |
| Format | WAV (uncompressed) |
| Completeness | 48/51 subjects with full 20 clips |

The `.lab` files are still uninspected — next cell examines their content, which turns out to be critical for the processing pipeline.

### VOICE-1 Results: Time-aligned digit transcriptions (HTK format)

The `.lab` files contain **time-aligned transcriptions** in HTK format:

In [23]:
# ============================================================
# CELL VOICE-1: Inspect .lab content + G04/G10 split
# Confirms what the .lab files contain (transcription? timing?)
# and how the 20 clips split between the two sessions.
# ============================================================

from pathlib import Path

voice_path = BIOMEX_ROOT / "F001" / "F001" / "Voice"
files = sorted(voice_path.iterdir())

# ── Read a few .lab files fully ───────────────────────────────────
lab_files = [f for f in files if f.suffix.lower() == '.lab']
print("=== Full content of 3 .lab files ===\n")
for lf in lab_files[:3]:
    with open(lf, encoding='utf-8', errors='ignore') as fh:
        content = fh.read()
    print(f"  --- {lf.name} ---")
    print(f"  {repr(content)}")   # repr shows hidden chars / structure
    print()

# ── Split clips by session ────────────────────────────────────────
wav_files = [f for f in files if f.suffix.lower() == '.wav']
g04 = [f for f in wav_files if 'G04' in f.name]
g10 = [f for f in wav_files if 'G10' in f.name]
print(f"=== Session split (F001) ===")
print(f"  G04 clips: {len(g04)}  →  {[f.stem.split('_')[-1] for f in g04]}")
print(f"  G10 clips: {len(g10)}  →  {[f.stem.split('_')[-1] for f in g10]}")

# ── Check duration consistency across a few clips ────────────────
import soundfile as sf
print(f"\n=== Duration check across clips ===")
for f in wav_files[:6]:
    info = sf.info(f)
    print(f"  {f.name:30s}  {info.duration:5.2f}s  {info.samplerate}Hz")

=== Full content of 3 .lab files ===

  --- F001_01G04_1.lab ---
  '0 8967499 SILENCIO \n8967500 11548125 UN0 \n11548126 24926874 SILENCIO \n24926875 27385625 DOS \n27385626 45953124 SILENCIO \n45953125 49636875 TRES \n49636876 65763124 SILENCIO \n65763125 69707500 CUATRO \n69707501 80000625 SILENCIO \n'

  --- F001_01G04_10.lab ---
  '0 5200624 SILENCIO \n5200625 9003750 OCHO \n9003751 25694374 SILENCIO \n25694375 28874375 UN0 \n28874376 45966874 SILENCIO \n45966875 49698750 CINCO \n49698751 66308749 SILENCIO \n66308750 69272500 CERO \n69272501 80000625 SILENCIO \n'

  --- F001_01G04_2.lab ---
  '0 5.189374e+06 SILENCIO \n5.189375e+06 9685625 CINCO \n9685626 26566249 SILENCIO \n26566250 29028125 TRES \n29028126 4.674937e+07 SILENCIO \n4.674938e+07 50453125 DOS \n50453126 66402499 SILENCIO \n66402500 70131875 NUEVE \n70131876 80000625 SILENCIO \n'

=== Session split (F001) ===
  G04 clips: 10  →  ['1', '10', '2', '3', '4', '5', '6', '7', '8', '9']
  G10 clips: 10  →  ['1', '10', '2', '

- Times are in **100-nanosecond units** (divide by 1e7 for seconds)
- Labels are Spanish digit names (UNO, DOS, TRES...) or SILENCIO
- Each clip contains 4 digits (G04) or 10 digits (G10) separated by ~1.5s silences

**Example (F001_01G04_1.lab):**

| Start (s) | End (s) | Label |
|-----------|---------|-------|
| 0.000 | 0.897 | SILENCIO |
| 0.897 | 1.155 | UNO |
| 1.155 | 2.493 | SILENCIO |
| 2.493 | 2.739 | DOS |

**Data quality issues found:**
- Some files use `UN0` (with zero) instead of `UNO` (with letter O) — corrected in the parser
- Some timestamps use scientific notation (e.g. `5.189e+06`) — parser handles both formats

**Why this matters:** each 8-second clip is ~80% silence. Without the `.lab` alignment, speaker embeddings would be computed mostly on silence, degrading their quality. The `.lab` files allow us to extract *only* the spoken segments.

## VOICE-2: Voice data completeness audit

Before processing all subjects, we verify that every subject has the expected voice files — the same quality control step applied to EEG data in Part 1.

For each subject we check:
- Existence of the Voice folder
- Number of `.wav` files (expected: 20)
- Session split (expected: 10 G04 + 10 G10)
- Duration consistency across clips

In [24]:
# ============================================================
# CELL VOICE-2: Audit voice data completeness across subjects
# Checks every subject has the expected clips, flags anomalies
# in count, duration, or sample rate — same spirit as the EDF
# audit we did in Phase 1.
# ============================================================

import soundfile as sf

print("=== Voice data audit (all subjects) ===\n")
print(f"  {'Subject':8s}  {'#wav':>5s}  {'G04':>4s}  {'G10':>4s}  "
      f"{'dur range':>14s}  status")
print("  " + "-" * 56)

voice_summary = []
for sid in metadata['subject_id'].values:
    vp = BIOMEX_ROOT / sid / sid / "Voice"
    if not vp.exists():
        print(f"  {sid:8s}  {'—':>5s}  {'—':>4s}  {'—':>4s}  "
              f"{'—':>14s}  NO VOICE FOLDER")
        voice_summary.append({'subject_id': sid, 'n_wav': 0})
        continue

    wavs = sorted(vp.glob("*.wav"))
    g04  = [w for w in wavs if 'G04' in w.name]
    g10  = [w for w in wavs if 'G10' in w.name]

    durs = []
    for w in wavs:
        try:
            durs.append(sf.info(w).duration)
        except Exception:
            pass

    dur_str = f"{min(durs):.1f}-{max(durs):.1f}s" if durs else "n/a"
    status  = "OK" if len(wavs) == 20 else f"⚠ {len(wavs)} clips"

    print(f"  {sid:8s}  {len(wavs):>5d}  {len(g04):>4d}  {len(g10):>4d}  "
          f"{dur_str:>14s}  {status}")

    voice_summary.append({'subject_id': sid, 'n_wav': len(wavs),
                          'n_g04': len(g04), 'n_g10': len(g10)})

voice_audit = pd.DataFrame(voice_summary)
print(f"\n=== Summary ===")
print(f"  Subjects with full 20 clips : "
      f"{(voice_audit['n_wav'] == 20).sum()} / {len(voice_audit)}")
print(f"  Subjects with missing clips : "
      f"{(voice_audit['n_wav'] < 20).sum()}")
print(f"  Subjects with no voice      : "
      f"{(voice_audit['n_wav'] == 0).sum()}")

=== Voice data audit (all subjects) ===

  Subject    #wav   G04   G10       dur range  status
  --------------------------------------------------------
  F001         20    10    10       8.0-20.0s  OK
  F002         20    10    10       8.0-20.0s  OK
  F003         20    10    10       8.0-20.0s  OK
  F004         20    10    10       8.0-20.0s  OK
  F005         20    10    10       8.0-20.0s  OK
  F006         20    10    10       8.0-20.0s  OK
  F007         20    10    10       8.0-20.0s  OK
  F008         20    10    10       8.0-20.0s  OK
  F009         20    10    10       8.0-20.0s  OK
  F010         20    10    10       8.0-20.0s  OK
  F011         20    10    10       8.0-20.0s  OK
  F012         20    10    10       8.0-20.0s  OK
  F013         20    10    10       8.0-20.0s  OK
  F014         20    10    10       8.0-20.0s  OK
  F015         20    10    10       8.0-20.0s  OK
  F016         20    10    10       8.0-20.0s  OK
  F017         20    10    10       8.0-20.0s 

### VOICE-2 Results: Complete dataset

| Check | Result |
|-------|--------|
| Subjects with full 20 clips | 48 / 51 |
| Subjects with missing clips | 0 |
| Subjects with no voice folder | 0 |
| G04 clip duration | ~8 s (consistent) |
| G10 clip duration | ~20 s (consistent) |

The voice dataset is more complete than the EEG dataset (where F008 and M001 had problematic recordings). No subjects need to be excluded from voice processing.

## VOICE-3: Parse alignment files + extract speech segments

### The silence problem

Each 8-second clip contains only ~4 short digit utterances (~0.3–0.5 s each) separated by long silences (~1.5 s each). On average, **only ~20% of each clip is actual speech**.

If we computed speaker embeddings on the full 8-second clips, the embeddings would mostly represent silence — not the speaker's vocal identity. We need to extract *only* the spoken segments.

### Solution: use the .lab alignment files

The `.lab` files tell us exactly where each digit starts and ends. We:
1. Parse the `.lab` file (handling scientific notation and the `UN0` → `UNO` typo)
2. Load the `.wav` audio
3. Cut out only the segments labelled as digits (not SILENCIO)
4. Add ±50 ms padding around each segment to avoid clipping speech onsets/offsets
5. Concatenate all speech segments into a single continuous audio array

This gives us a clean, silence-free speech signal per clip.

In [25]:
# ============================================================
# CELL VOICE-3: Parse .lab alignment files + extract speech
#
# .lab format (HTK): each line is "start end label"
#   - times are in 100-nanosecond units → divide by 1e7 for seconds
#   - labels: digit words (UNO, DOS...) or SILENCIO
#   - some files use scientific notation (5.18e+06) → parse as float
#   - some files have OCR-style typos: UN0 (zero) instead of UNO
#
# We extract only the spoken-digit segments, discarding silence,
# so speaker embeddings are computed on voice, not on silence.
# ============================================================

import soundfile as sf
import numpy as np

# Normalisation map for digit-label typos / variants
DIGIT_FIX = {
    'UN0': 'UNO', 'UNO': 'UNO', 'DOS': 'DOS', 'TRES': 'TRES',
    'CUATRO': 'CUATRO', 'CINCO': 'CINCO', 'SEIS': 'SEIS',
    'SIETE': 'SIETE', 'OCHO': 'OCHO', 'NUEVE': 'NUEVE',
    'CERO': 'CERO', 'DIEZ': 'DIEZ',
}
HTK_UNIT = 1e7   # 100-ns units per second


def parse_lab(lab_path):
    """
    Parse an HTK .lab file into a list of (start_s, end_s, label).
    Times converted to seconds. Labels normalised. Silence kept
    (flagged) so we can verify coverage, but caller can filter it.
    """
    segments = []
    with open(lab_path, encoding='utf-8', errors='ignore') as fh:
        for line in fh:
            parts = line.split()
            if len(parts) < 3:
                continue
            start = float(parts[0]) / HTK_UNIT      # float() handles 5.1e+06
            end   = float(parts[1]) / HTK_UNIT
            label = parts[2].strip().upper()
            label = DIGIT_FIX.get(label, label)     # fix UN0 → UNO
            segments.append((start, end, label))
    return segments


def extract_speech(wav_path, lab_path, sr_expected=16000, pad_s=0.05):
    """
    Load a wav and return only the concatenated spoken-digit audio
    (silence removed), plus the list of digits spoken in order.

    pad_s: small padding (s) added around each segment so we don't
           clip the onset/offset of each digit.
    """
    audio, sr = sf.read(wav_path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)          # force mono
    segments = parse_lab(lab_path)

    speech_chunks = []
    digits        = []
    for start, end, label in segments:
        if label == 'SILENCIO':
            continue
        i0 = max(0, int((start - pad_s) * sr))
        i1 = min(len(audio), int((end + pad_s) * sr))
        speech_chunks.append(audio[i0:i1])
        digits.append(label)

    speech = np.concatenate(speech_chunks) if speech_chunks else np.array([])
    return speech, digits, sr


# ── Test on F001 clip 1 ───────────────────────────────────────────
voice_path = BIOMEX_ROOT / "F001" / "F001" / "Voice"
wav = voice_path / "F001_01G04_1.wav"
lab = voice_path / "F001_01G04_1.lab"

segs = parse_lab(lab)
print("=== Parsed segments (F001_01G04_1) ===")
for s, e, l in segs:
    tag = '   (silence)' if l == 'SILENCIO' else f'   → DIGIT: {l}'
    print(f"  {s:6.3f}s – {e:6.3f}s  {l:10s}{tag}")

speech, digits, sr = extract_speech(wav, lab)
full_audio, _ = sf.read(wav)
print(f"\n=== Speech extraction ===")
print(f"  Digits spoken    : {digits}")
print(f"  Full clip length : {len(full_audio)/sr:.2f}s")
print(f"  Speech-only length: {len(speech)/sr:.2f}s  "
      f"({100*len(speech)/len(full_audio):.0f}% of clip)")
print(f"  Silence removed  : {(len(full_audio)-len(speech))/sr:.2f}s")

=== Parsed segments (F001_01G04_1) ===
   0.000s –  0.897s  SILENCIO     (silence)
   0.897s –  1.155s  UNO          → DIGIT: UNO
   1.155s –  2.493s  SILENCIO     (silence)
   2.493s –  2.739s  DOS          → DIGIT: DOS
   2.739s –  4.595s  SILENCIO     (silence)
   4.595s –  4.964s  TRES         → DIGIT: TRES
   4.964s –  6.576s  SILENCIO     (silence)
   6.576s –  6.971s  CUATRO       → DIGIT: CUATRO
   6.971s –  8.000s  SILENCIO     (silence)

=== Speech extraction ===
  Digits spoken    : ['UNO', 'DOS', 'TRES', 'CUATRO']
  Full clip length : 8.00s
  Speech-only length: 1.67s  (21% of clip)
  Silence removed  : 6.33s


### VOICE-3 Results: Speech extraction verified on F001

| Metric | Value |
|--------|-------|
| Digits detected | 4 (UNO, DOS, TRES, CUATRO) |
| Full clip duration | 8.00 s |
| Speech-only duration | 1.67 s (21% of clip) |
| Silence removed | 6.33 s (79% of clip) |

The parser correctly:
- Converted times from HTK units (100 ns) to seconds
- Fixed the `UN0` → `UNO` typo
- Extracted only voiced segments with ±50 ms padding

**Key insight:** without silence removal, ~80% of the signal fed to the speaker encoder would be silence. This step is essential for meaningful voice embeddings.

## VOICE-4: Load the speaker embedding model

### What is a speaker embedding?

A speaker embedding is a compact numerical vector (192 dimensions) that captures a person's **vocal identity** — their timbre, pitch, speaking style — independent of what words they say.

This is the voice equivalent of the EEG embedding from Phase 1:

| Modality | Model | Embedding dim | Captures |
|----------|-------|---------------|----------|
| EEG | EEGNet | 32 | Neural fingerprint |
| Voice | ECAPA-TDNN | 192 | Vocal fingerprint |

Phase 3 will learn the mapping between these two representations.

### Model choice: ECAPA-TDNN (SpeechBrain)

**ECAPA-TDNN** (Desplanques et al., 2020) is the current state-of-the-art for speaker recognition. It is:
- Pretrained on **VoxCeleb** (thousands of speakers) — no fine-tuning needed
- Based on PyTorch (already installed for EEGNet)
- Produces **192-dimensional** L2-normalised speaker embeddings

Note: the first attempt (Resemblyzer) failed due to a missing C++ compiler on Windows. ECAPA-TDNN avoids this because it is pure Python on top of PyTorch. A symlink permission issue on Windows (`WinError 1314`) was resolved by forcing SpeechBrain to copy files instead of symlinking them (`LocalStrategy.COPY`).

In [ ]:
# ============================================================
# CELL VOICE-4 (v4): Force SpeechBrain to COPY, not symlink
#                    Install + load the speaker embedding model
# Resemblyzer = speaker encoder from Real-Time-Voice-Cloning.
#   - 256-dim speaker embeddings
#   - runs fine on CPU
#   - designed for voice cloning (matches your Phase 3 goal)
# SpeechBrain (not HF) is creating the symlink. Newer versions
# accept a LocalStrategy to copy files instead.
# ============================================================

import os
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

import torch
from speechbrain.inference.speaker import EncoderClassifier

spk_encoder = None

# --- Attempt 1: explicit COPY strategy (newer SpeechBrain) ---
try:
    from speechbrain.utils.fetching import LocalStrategy
    spk_encoder = EncoderClassifier.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir=str(PROJECT_ROOT / "pretrained_ecapa"),
        run_opts={"device": "cpu"},
        local_strategy=LocalStrategy.COPY,
    )
    print("✓ Loaded with LocalStrategy.COPY")
except Exception as e1:
    print(f"  Attempt 1 (COPY strategy) failed: {type(e1).__name__}")

    # --- Attempt 2: load straight from the HF cache, no savedir copy ---
    try:
        from huggingface_hub import snapshot_download
        cache_dir = snapshot_download(
            "speechbrain/spkrec-ecapa-voxceleb",
            local_dir_use_symlinks=False,
        )
        spk_encoder = EncoderClassifier.from_hparams(
            source=cache_dir,
            savedir=cache_dir,           # same dir → nothing to symlink
            run_opts={"device": "cpu"},
        )
        print("✓ Loaded directly from HF cache (no symlink needed)")
    except Exception as e2:
        print(f"  Attempt 2 (cache load) failed: {type(e2).__name__}: {e2}")

if spk_encoder is not None:
    print(f"\n✓ ECAPA-TDNN ready  |  embedding dim: 192")
else:
    print("\n✗ Both attempts failed — see decision below.")

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

✓ Loaded with LocalStrategy.COPY

✓ ECAPA-TDNN ready  |  embedding dim: 192


### VOICE-4 Results

✓ ECAPA-TDNN speaker encoder loaded successfully.

| Property | Value |
|----------|-------|
| Model | ECAPA-TDNN (SpeechBrain) |
| Pretrained on | VoxCeleb |
| Embedding dimension | 192 |
| Device | CPU |
| Input format | 16 kHz mono (matches BIOMEX audio) |

The encoder is ready to convert speech segments into speaker identity vectors. Next step: verify that these embeddings actually capture identity (sanity check).

## VOICE-5: Does the embedding capture speaker identity?

### Sanity check before processing all subjects

Before investing time in embedding all 51 subjects, we verify the fundamental property that speaker embeddings must satisfy:

> **Two clips from the same person must be more similar to each other than two clips from different people.**

If this fails, the embedding is useless for voice identity — regardless of how sophisticated the model is.

### Method

- Embed 3 clips each from F001 and F002 (6 embeddings total)
- Compute pairwise **cosine similarity** (dot product of L2-normalised vectors, range [−1, +1])
- Compare **within-subject** similarity (same person) vs **between-subject** similarity (different people)
- The **gap** (within − between) quantifies how well the embedding separates identities

### Note on speech duration

Each clip yields only ~1.5–1.8 seconds of speech after silence removal. ECAPA-TDNN ideally prefers several seconds of audio. With short segments, embeddings are noisier — we expect moderate within-subject similarity (~0.6–0.8), not perfect (~1.0).

In [30]:
# ============================================================
# CELL VOICE-5: Embedding sanity check — does it capture identity?
#
# We embed several clips from 2 subjects and check that
# within-subject similarity > between-subject similarity.
# Similarity = cosine similarity between L2-normalised embeddings.
#
# ECAPA expects 16 kHz mono audio (which is what BIOMEX has).
# We feed the SILENCE-REMOVED speech from CELL VOICE-3.
# ============================================================

import torch
import numpy as np

def embed_speech(speech, sr=16000):
    """Return a 192-dim ECAPA embedding for a 1-D speech array."""
    wav = torch.tensor(speech, dtype=torch.float32).unsqueeze(0)  # (1, n)
    with torch.no_grad():
        emb = spk_encoder.encode_batch(wav)        # (1, 1, 192)
    emb = emb.squeeze().cpu().numpy()              # (192,)
    return emb / (np.linalg.norm(emb) + 1e-12)     # L2-normalise

def cosine(a, b):
    return float(np.dot(a, b))   # already normalised → dot = cosine

# ── Embed 3 clips each from F001 and F002 ─────────────────────────
def clip_paths(sid, n):
    vp = BIOMEX_ROOT / sid / sid / "Voice"
    return (vp / f"{sid}_01G04_{n}.wav", vp / f"{sid}_01G04_{n}.lab")

embeddings = {}
for sid in ['F001', 'F002']:
    for n in [1, 2, 3]:
        wav, lab = clip_paths(sid, n)
        speech, digits, sr = extract_speech(wav, lab)
        embeddings[f"{sid}_{n}"] = embed_speech(speech, sr)
        print(f"  Embedded {sid} clip {n}  "
              f"(speech {len(speech)/sr:.2f}s, digits {digits})")

# ── Similarity matrix ─────────────────────────────────────────────
keys = list(embeddings.keys())
print(f"\n=== Cosine similarity matrix ===")
print(f"  {'':10s}" + "".join(f"{k:>10s}" for k in keys))
for k1 in keys:
    row = f"  {k1:10s}"
    for k2 in keys:
        row += f"{cosine(embeddings[k1], embeddings[k2]):>10.3f}"
    print(row)

# ── Within vs between summary ─────────────────────────────────────
within, between = [], []
for i, k1 in enumerate(keys):
    for j, k2 in enumerate(keys):
        if j <= i:
            continue
        sim = cosine(embeddings[k1], embeddings[k2])
        same = k1.split('_')[0] == k2.split('_')[0]
        (within if same else between).append(sim)

print(f"\n=== Identity check ===")
print(f"  Within-subject  similarity (same person) : "
      f"mean {np.mean(within):.3f}  (n={len(within)})")
print(f"  Between-subject similarity (diff people) : "
      f"mean {np.mean(between):.3f}  (n={len(between)})")
gap = np.mean(within) - np.mean(between)
print(f"  Gap (within − between)                    : {gap:+.3f}")
print(f"\n  → {'GOOD' if gap > 0.1 else 'WEAK' if gap > 0 else 'BAD'}: "
      f"embedding {'captures identity well' if gap > 0.1 else 'captures identity weakly' if gap > 0 else 'does NOT capture identity'}")

  Embedded F001 clip 1  (speech 1.67s, digits ['UNO', 'DOS', 'TRES', 'CUATRO'])
  Embedded F001 clip 2  (speech 1.84s, digits ['CINCO', 'TRES', 'DOS', 'NUEVE'])
  Embedded F001 clip 3  (speech 1.53s, digits ['UNO', 'CERO', 'SIETE', 'TRES'])
  Embedded F002 clip 1  (speech 1.32s, digits ['UNO', 'DOS', 'TRES', 'CUATRO'])
  Embedded F002 clip 2  (speech 1.21s, digits ['CINCO', 'TRES', 'DOS', 'NUEVE'])
  Embedded F002 clip 3  (speech 1.23s, digits ['UNO', 'CERO', 'SIETE', 'TRES'])

=== Cosine similarity matrix ===
                F001_1    F001_2    F001_3    F002_1    F002_2    F002_3
  F001_1         1.000     0.803     0.798     0.107     0.136     0.246
  F001_2         0.803     1.000     0.776     0.073     0.145     0.200
  F001_3         0.798     0.776     1.000     0.147     0.200     0.277
  F002_1         0.107     0.073     0.147     1.000     0.670     0.647
  F002_2         0.136     0.145     0.200     0.670     1.000     0.641
  F002_3         0.246     0.200     0.277    

### VOICE-5 Results: Embedding captures identity clearly

| Metric | Value |
|--------|-------|
| Within-subject similarity (same person) | 0.722 |
| Between-subject similarity (diff people) | 0.170 |
| **Gap (within − between)** | **+0.552** |

The gap of **+0.55 is excellent**, especially given only ~1.7 seconds of speech per clip. For comparison, a gap below 0.1 would be concerning, and above 0.3 is considered strong.

**Key observation:** F001 clip 1 and F002 clip 1 contain the same digits (UNO, DOS, TRES, CUATRO) but their similarity is only 0.107. This confirms the embedding captures **who speaks**, not **what is said** — exactly the property needed for Phase 3.

The speaker encoder is validated. Next step: compute embeddings for all subjects.

## VOICE-6: Build per-subject voice embeddings

### From clip-level to subject-level

VOICE-5 showed that individual clip embeddings (~1.7 s of speech each) capture identity but with some noise. To get a cleaner, more stable representation, we **average** the embeddings across all 10 clips of each session.

This mirrors the EEG approach exactly:

| Step | EEG (Phase 1) | Voice (this section) |
|------|---------------|---------------------|
| Raw signal | 2 s EEG epoch | ~1.7 s speech clip |
| Per-sample embedding | EEGNet → 32 dim | ECAPA → 192 dim |
| Per-subject embedding | Mean of ~100 epochs | Mean of 10 clips |
| Re-normalisation | L2 | L2 |

### Cross-session split

Sessions are kept separate to enable cross-session validation:
- **G10 embeddings** → primary (training reference for Phase 3)
- **G04 embeddings** → held-out (cross-session validation)

This is the same strategy used for EEG: train on G10, validate on G04.

In [31]:
# ============================================================
# CELL VOICE-6: Build per-subject voice embeddings (all subjects)
#
# For each subject:
#   - extract silence-removed speech from each clip
#   - embed each clip with ECAPA (192-dim)
#   - average clip embeddings → one voice fingerprint per subject
#
# Sessions kept separate (mirrors the EEG G10/G04 split):
#   G10 → primary embeddings (training reference for Phase 3)
#   G04 → held-out embeddings (cross-session validation)
#
# Output: two DataFrames (G10, G04), one row per subject, 192 cols.
# ============================================================

import numpy as np
import time

def subject_voice_embedding(sid, session='G10'):
    """
    Average ECAPA embedding over all valid clips of one session.
    Returns (embedding_192, n_clips_used) or (None, 0) if no audio.
    """
    vp = BIOMEX_ROOT / sid / sid / "Voice"
    if not vp.exists():
        return None, 0

    clip_embs = []
    for wav in sorted(vp.glob(f"*{session}*.wav")):
        lab = wav.with_suffix('.lab')
        if not lab.exists():
            continue
        try:
            speech, digits, sr = extract_speech(wav, lab)
            if len(speech) < 0.3 * sr:        # skip clips with <0.3s speech
                continue
            clip_embs.append(embed_speech(speech, sr))
        except Exception:
            continue

    if not clip_embs:
        return None, 0

    mean_emb = np.mean(clip_embs, axis=0)
    mean_emb = mean_emb / (np.linalg.norm(mean_emb) + 1e-12)  # re-normalise
    return mean_emb, len(clip_embs)

# ── Process every subject, both sessions ──────────────────────────
print("Building per-subject voice embeddings (ECAPA-TDNN)...")
print("This runs the encoder on ~20 clips × 51 subjects — a few minutes.\n")
print(f"  {'Subject':8s}  {'G10 clips':>9s}  {'G04 clips':>9s}  status")
print("  " + "-" * 42)

g10_rows, g04_rows = [], []
t0 = time.time()

for sid in metadata['subject_id'].values:
    emb10, n10 = subject_voice_embedding(sid, 'G10')
    emb04, n04 = subject_voice_embedding(sid, 'G04')

    status = "OK" if (emb10 is not None and emb04 is not None) else "⚠ partial"
    if emb10 is None and emb04 is None:
        status = "✗ no audio"

    print(f"  {sid:8s}  {n10:>9d}  {n04:>9d}  {status}")

    if emb10 is not None:
        row = {'subject_id': sid}
        row.update({f'voice_{i:03d}': v for i, v in enumerate(emb10)})
        g10_rows.append(row)
    if emb04 is not None:
        row = {'subject_id': sid}
        row.update({f'voice_{i:03d}': v for i, v in enumerate(emb04)})
        g04_rows.append(row)

# ── Assemble + merge metadata ─────────────────────────────────────
voice_g10 = pd.DataFrame(g10_rows).merge(metadata, on='subject_id')
voice_g04 = pd.DataFrame(g04_rows).merge(metadata, on='subject_id')

print(f"\n=== Voice embedding matrices ===")
print(f"  G10: {voice_g10.shape[0]} subjects × 192 voice dims")
print(f"  G04: {voice_g04.shape[0]} subjects × 192 voice dims")
print(f"  Time elapsed: {time.time()-t0:.0f}s")

# ── Save to disk (input for Phase 3) ──────────────────────────────
voice_g10.to_csv(PROJECT_ROOT / 'voice_embeddings_G10.csv', index=False)
voice_g04.to_csv(PROJECT_ROOT / 'voice_embeddings_G04.csv', index=False)
print(f"\n✓ Saved:")
print(f"    voice_embeddings_G10.csv")
print(f"    voice_embeddings_G04.csv")

Building per-subject voice embeddings (ECAPA-TDNN)...
This runs the encoder on ~20 clips × 51 subjects — a few minutes.

  Subject   G10 clips  G04 clips  status
  ------------------------------------------
  F001             10         10  OK
  F002             10         10  OK
  F003             10         10  OK
  F004             10         10  OK
  F005             10         10  OK
  F006             10         10  OK
  F007             10         10  OK
  F008             10         10  OK
  F009             10         10  OK
  F010             10         10  OK
  F011             10         10  OK
  F012             10         10  OK
  F013             10         10  OK
  F014             10         10  OK
  F015             10         10  OK
  F016             10         10  OK
  F017             10         10  OK
  F018             10         10  OK
  F019             10         10  OK
  F020             10         10  OK
  F021             10         10  OK
  F022          

### VOICE-6 Results

| Session | Subjects | Clips per subject | Embedding dim |
|---------|----------|-------------------|---------------|
| G10 | 51 | 10 | 192 |
| G04 | 51 | 10 | 192 |

All subjects processed successfully with zero errors — more complete than the EEG dataset.

**Saved to disk:**
- `voice_embeddings_G10.csv` — primary embeddings (51 subjects × 192 dimensions + metadata)
- `voice_embeddings_G04.csv` — held-out embeddings (51 subjects × 192 dimensions + metadata)

These two files, together with the EEG embeddings from Phase 1 (`eeg_embeddings.csv`), are the inputs for Phase 3: learning the mapping from neural fingerprint to vocal fingerprint.

## VOICE-7: Cross-session speaker identification (G10 → G04)

### The definitive test

This is the voice equivalent of the EEGNet cross-session test from Phase 1. The question is identical:

> If we take a subject's G10 embedding and search for the closest match among all G04 embeddings, do we correctly identify the same person?

This is a **much harder test** than within-session similarity (VOICE-5): the model has never seen any G04 data — it must rely purely on the stability of vocal identity across recording sessions.

### Method

- Compute the **51×51 cosine similarity matrix** (G10 rows × G04 columns)
- For each subject (row), rank all G04 embeddings by similarity
- **Top-1 accuracy**: is the closest match the correct person?
- **Top-3 / Top-5**: is the correct person among the top 3 or 5 matches?
- Report any misidentifications (who was confused with whom)

### Expected result

Voice is far more discriminative than EEG for identity. EEGNet achieved 73.1% cross-session accuracy with low-resolution EEG. ECAPA-TDNN with 16 kHz speech should achieve **>90%**, providing a strong anchor for Phase 3.

In [32]:
# ============================================================
# CELL VOICE-7: Cross-session speaker identification (G10 → G04)
#
# For each subject: find which G04 embedding is most similar
# (cosine) to their G10 embedding. If the closest match is
# themselves → correct identification.
#
# This mirrors the EEGNet cross-session test (73.1% on EEG).
# Voice should be much higher — speaker identity is more
# stable than neural patterns across sessions.
#
# Also: rank-based analysis (Top-1, Top-3, Top-5 accuracy).
# ============================================================

import numpy as np

# ── Extract embedding matrices (aligned by subject order) ────────
voice_cols = [c for c in voice_g10.columns if c.startswith('voice_')]

# Only subjects present in BOTH sessions
common_sids = sorted(set(voice_g10['subject_id']) & set(voice_g04['subject_id']))

E_g10 = (voice_g10.set_index('subject_id')
                   .loc[common_sids, voice_cols].values)   # (N, 192)
E_g04 = (voice_g04.set_index('subject_id')
                   .loc[common_sids, voice_cols].values)   # (N, 192)

N = len(common_sids)
print(f"=== Cross-session speaker identification ===")
print(f"  Subjects in both sessions: {N}")
print(f"  Chance level (random): {1/N*100:.1f}%\n")

# ── Cosine similarity matrix (G10 × G04) ─────────────────────────
# Rows = G10 subjects, Cols = G04 subjects
# Diagonal should be the highest in each row if identification works
sim_matrix = E_g10 @ E_g04.T   # both already L2-normalised

# ── Top-K accuracy ────────────────────────────────────────────────
top1, top3, top5 = 0, 0, 0
per_subject = []

for i in range(N):
    ranked = np.argsort(-sim_matrix[i])   # indices sorted by descending sim
    rank   = np.where(ranked == i)[0][0]  # rank of the true match (0-indexed)

    top1 += (rank == 0)
    top3 += (rank < 3)
    top5 += (rank < 5)

    per_subject.append({
        'subject_id': common_sids[i],
        'rank':       rank + 1,              # 1-indexed for readability
        'self_sim':   sim_matrix[i, i],
        'best_sim':   sim_matrix[i, ranked[0]],
        'best_match': common_sids[ranked[0]],
        'correct':    rank == 0,
    })

print(f"  Top-1 accuracy: {top1}/{N}  ({100*top1/N:.1f}%)")
print(f"  Top-3 accuracy: {top3}/{N}  ({100*top3/N:.1f}%)")
print(f"  Top-5 accuracy: {top5}/{N}  ({100*top5/N:.1f}%)")

# ── Show misidentifications (if any) ─────────────────────────────
errors = [r for r in per_subject if not r['correct']]
if errors:
    print(f"\n=== Misidentified subjects ({len(errors)}) ===")
    print(f"  {'Subject':8s}  {'Rank':>5s}  {'Self sim':>9s}  "
          f"{'Best sim':>9s}  {'Confused with'}")
    print("  " + "-" * 52)
    for r in sorted(errors, key=lambda x: x['rank']):
        print(f"  {r['subject_id']:8s}  {r['rank']:>5d}  "
              f"{r['self_sim']:>9.3f}  {r['best_sim']:>9.3f}  "
              f"{r['best_match']}")
else:
    print(f"\n  PERFECT — every subject correctly identified across sessions.")

# ── Diagonal vs off-diagonal summary ──────────────────────────────
diag     = np.diag(sim_matrix)
off_diag = sim_matrix[~np.eye(N, dtype=bool)]
print(f"\n=== Similarity distribution ===")
print(f"  Same-person  (diagonal)    : "
      f"mean {diag.mean():.3f}  min {diag.min():.3f}  max {diag.max():.3f}")
print(f"  Diff-person  (off-diagonal): "
      f"mean {off_diag.mean():.3f}  min {off_diag.min():.3f}  max {off_diag.max():.3f}")
print(f"  Gap (same − diff mean)     : {diag.mean()-off_diag.mean():+.3f}")

=== Cross-session speaker identification ===
  Subjects in both sessions: 51
  Chance level (random): 2.0%

  Top-1 accuracy: 51/51  (100.0%)
  Top-3 accuracy: 51/51  (100.0%)
  Top-5 accuracy: 51/51  (100.0%)

  PERFECT — every subject correctly identified across sessions.

=== Similarity distribution ===
  Same-person  (diagonal)    : mean 0.942  min 0.861  max 0.987
  Diff-person  (off-diagonal): mean 0.198  min -0.066  max 0.613
  Gap (same − diff mean)     : +0.744


### VOICE-7 Results: 100% cross-session speaker identification

| Metric | Voice (ECAPA) | EEG (EEGNet) |
|--------|---------------|--------------|
| Top-1 accuracy | **100%** (51/51) | 73.1% |
| Same-person similarity | 0.942 | — |
| Diff-person similarity | 0.198 | — |
| Gap (same − diff) | **+0.744** | — |

Every single subject was correctly identified across sessions — zero errors. The minimum same-person similarity (0.861) is still far above the maximum different-person similarity (0.613), meaning there is no ambiguous case.

### Comparison: voice vs EEG for identity

Voice identification is near-perfect (100%), while EEG identification is strong but imperfect (73.1%). This gap is expected — voice is a direct expression of identity (unique vocal tract anatomy), while EEG is an indirect, noisy measurement through the skull.

However, this gap is precisely what makes Phase 3 meaningful: if both modalities identified equally well, there would be no added value in mapping between them. The fact that EEG carries identity information (73%) but less cleanly than voice (100%) means that **learning the EEG → voice mapping could assign a personalised voice to someone whose EEG we can read but whose voice we cannot hear.**

### Summary: voice processing pipeline complete

| Step | Output | Saved to disk |
|------|--------|---------------|
| Speech extraction | Silence-free audio per clip | — |
| Speaker embedding | 192-dim ECAPA vector per clip | — |
| Subject-level embedding | Mean of 10 clips per subject | `voice_embeddings_G10.csv` |
| Cross-session held-out | Separate G04 embeddings | `voice_embeddings_G04.csv` |

Both embedding files are ready as inputs for Phase 3: learning the mapping from EEG neural fingerprint (32 dim) to voice fingerprint (192 dim).